# Deep Learning Diagnostics Colab
Spaced representation snapshots; Drive stores analysis only, checkpoints stay in /content.

In [ ]:
!pip -q install umap-learn tensorboard
import json,random
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader,Subset
from torchvision import datasets,transforms
from torch.utils.tensorboard import SummaryWriter
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import umap.umap_ as umap
from google.colab import drive
D=torch.device('cuda' if torch.cuda.is_available() else 'cpu');S=7;FAST=True
random.seed(S);np.random.seed(S);torch.manual_seed(S)
drive.mount('/content/drive');R=Path('/content/drive/MyDrive/deep_learning_diagnostics');C=R/'csv';N=R/'npz';G=R/'figures';T=R/'tensorboard';J=R/'summaries';K=Path('/content/local_checkpoints')
for p in [C,N,G,T,J,K]:p.mkdir(parents=True,exist_ok=True)
trf=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize((.4914,.4822,.4465),(.247,.2435,.2616))]);evf=transforms.Compose([transforms.ToTensor(),transforms.Normalize((.4914,.4822,.4465),(.247,.2435,.2616))])
a=datasets.CIFAR10('/content/data',train=True,download=True,transform=trf);b=datasets.CIFAR10('/content/data',train=True,download=False,transform=evf);g=torch.Generator().manual_seed(S);ix=torch.randperm(len(a),generator=g).tolist();nt,nv,E=(12000,2000,6) if FAST else (40000,5000,12);ti,vi=ix[:nt],ix[nt:nt+nv];kw=dict(batch_size=256,num_workers=2,pin_memory=True);tl=DataLoader(Subset(a,ti),shuffle=True,**kw);te=DataLoader(Subset(b,ti),shuffle=False,**kw);vl=DataLoader(Subset(b,vi),shuffle=False,**kw);q=max(1,round(E/4));DE=sorted(set([0]+list(range(q,E+1,q))+[E]));print('diagnostic epochs',DE)
class M(nn.Module):
 def __init__(s):
  super().__init__();s.a=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU());s.b=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2));s.c=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2));s.d=nn.AdaptiveAvgPool2d(1);s.p=nn.Linear(128,64);s.h=nn.Linear(64,10)
 def forward(s,x,f=False):
  z={};x=s.a(x);z['stem']=x.mean((2,3));x=s.b(x);z['block1']=x.mean((2,3));x=s.c(x);z['block2']=x.mean((2,3));x=s.d(x).flatten(1);x=F.relu(s.p(x));z['penultimate']=x;o=s.h(x);return(o,z) if f else o
L=['stem','block1','block2','penultimate'];P={'stem':'a.0.weight','block1':'b.0.weight','block2':'c.0.weight','penultimate':'p.weight','head':'h.weight'};torch.manual_seed(S);I={k:v.cpu().clone() for k,v in M().state_dict().items()}
@torch.no_grad()
def eva(m,l):
 m.eval();ls=co=n=0
 for x,y in l:x,y=x.to(D),y.to(D);o=m(x);ls+=F.cross_entropy(o,y,reduction='sum').item();co+=(o.argmax(1)==y).sum().item();n+=len(y)
 return ls/n,co/n
def train(run,adam=False):
 m=M().to(D);m.load_state_dict(I);op=torch.optim.AdamW(m.parameters(),lr=2e-3) if adam else torch.optim.SGD(m.parameters(),lr=.08,momentum=.9);w=SummaryWriter(str(T/run));H=[];Y=[];torch.save(m.state_dict(),K/f'{run}_e0.pt');st=0
 for e in range(1,E+1):
  m.train();ls=co=n=0
  for x,y in tl:
   x,y=x.to(D),y.to(D);op.zero_grad();o=m(x);loss=F.cross_entropy(o,y);loss.backward();nm=dict(m.named_parameters());old={k:nm[v].detach().clone() for k,v in P.items()};r={'run':run,'epoch':e,'step':st}
   for k,v in P.items():r[k+'_grad']=nm[v].grad.norm().item()
   op.step();nm=dict(m.named_parameters())
   for k,v in P.items():r[k+'_uw']=(nm[v]-old[k]).norm().item()/(nm[v].norm().item()+1e-12);w.add_scalar('grad/'+k,r[k+'_grad'],st);w.add_scalar('update_weight/'+k,r[k+'_uw'],st)
   Y.append(r);st+=1;ls+=loss.item()*len(y);co+=(o.argmax(1)==y).sum().item();n+=len(y)
  va,ac=eva(m,vl);H.append([run,e,ls/n,co/n,va,ac]);w.add_scalar('val/loss',va,e);w.add_scalar('val/acc',ac,e)
  if e in DE:torch.save(m.state_dict(),K/f'{run}_e{e}.pt')
 w.close();pd.DataFrame(H,columns=['run','epoch','train_loss','train_acc','val_loss','val_acc']).to_csv(C/f'{run}_history.csv',index=False);pd.DataFrame(Y).to_csv(C/f'{run}_dynamics.csv',index=False);return m
sg=train('sgd');ad=train('adamw',True)
@torch.no_grad()
def feat(m,n=2000):
 z={k:[] for k in L};yy=[];pp=[];c=0;m.eval()
 for x,y in vl:
  o,f=m(x.to(D),True);u=min(len(y),n-c)
  for k in L:z[k].append(f[k][:u].cpu())
  yy.append(y[:u]);pp.append(o[:u].argmax(1).cpu());c+=u
  if c>=n:break
 return {k:torch.cat(v).numpy() for k,v in z.items()},torch.cat(yy).numpy(),torch.cat(pp).numpy()
def snap(run):
 d={}
 for e in DE:
  m=M().to(D);m.load_state_dict(torch.load(K/f'{run}_e{e}.pt',map_location='cpu'));f,y,p=feat(m);d[e]=(f,y,p);np.savez_compressed(N/f'{run}_e{e}.npz',y=y,pred=p,**f)
 return d
A=snap('sgd');B=snap('adamw');sf,y,p=A[E];af=B[E][0]
def er(x):
 x=torch.tensor(x).float();x-=x.mean(0);s=torch.linalg.svdvals(x);v=s*s;v/=v.sum();return float(torch.exp(-(v*torch.log(v.clamp_min(1e-12))).sum()))
def ck(x,y):
 x=torch.tensor(x).float();y=torch.tensor(y).float();x-=x.mean(0);y-=y.mean(0);return float((x.T@y).square().sum()/(((x.T@x).square().sum().sqrt()*(y.T@y).square().sum().sqrt())+1e-12))
rows=[]
for run,Z in [('sgd',A),('adamw',B)]:
 for e in DE:
  for l in L:rows.append([run,e,l,er(Z[e][0][l]),ck(Z[e][0][l],Z[0][0][l]),ck(Z[e][0][l],Z[E][0][l])])
pd.DataFrame(rows,columns=['run','epoch','layer','effective_rank','cka_init','cka_final']).to_csv(C/'representation_dynamics.csv',index=False)
for run,Z in [('sgd',A),('adamw',B)]:
 for e in DE:
  X=StandardScaler().fit_transform(Z[e][0]['penultimate'])
  for name,r in [('pca',PCA(2)),('umap',umap.UMAP(n_components=2,n_neighbors=20,min_dist=.15,random_state=S))]:
   z=r.fit_transform(X);df=pd.DataFrame({'x':z[:,0],'y':z[:,1],'label':Z[e][1],'pred':Z[e][2]});df.to_csv(C/f'{run}_{name}_e{e}.csv',index=False);plt.figure();plt.scatter(df.x,df.y,c=df.label,s=8,cmap='tab10');plt.title(f'{run} {name} e{e}');plt.savefig(G/f'{run}_{name}_e{e}.png',dpi=160);plt.close()
X=StandardScaler().fit_transform(sf['penultimate']);nnx=NearestNeighbors(n_neighbors=31).fit(X);_,ii=nnx.kneighbors(X);ld=[]
for i in range(len(X)):
 z=X[ii[i,1:]];z-=z.mean(0);s=np.linalg.svd(z,compute_uv=False);v=s*s;ld.append(np.searchsorted(np.cumsum(v)/v.sum(),.9)+1)
pd.DataFrame({'label':y,'pred':p,'local_dim90':ld}).to_csv(C/'local_dimension.csv',index=False)
w=SummaryWriter(str(T/'projector_sgd'));meta=[f'y={a} pred={b}' for a,b in zip(y,p)]
for l in L:w.add_embedding(torch.tensor(sf[l]).float(),metadata=meta,tag=l)
w.close()
# Hessian top eigenvalue
def htop(state):
 m=M().to(D);m.load_state_dict(state);x,y0=next(iter(te));x,y0=x[:128].to(D),y0[:128].to(D);ps=list(m.parameters());v=[torch.randn_like(a) for a in ps]
 for _ in range(8):
  n=torch.sqrt(sum((a*a).sum() for a in v));v=[a/n for a in v];g=torch.autograd.grad(F.cross_entropy(m(x),y0),ps,create_graph=True);hv=torch.autograd.grad(sum((a*b).sum() for a,b in zip(g,v)),ps);eig=sum((a*b).sum() for a,b in zip(v,hv)).item();v=[a.detach() for a in hv]
 return eig
hd=[['sgd_init',htop(torch.load(K/'sgd_e0.pt',map_location='cpu'))],['sgd_final',htop(torch.load(K/f'sgd_e{E}.pt',map_location='cpu'))],['adam_final',htop(torch.load(K/f'adamw_e{E}.pt',map_location='cpu'))]];pd.DataFrame(hd,columns=['condition','lambda_max']).to_csv(C/'hessian.csv',index=False)
# interpolation
def mix(a,b,t):return {k:((1-t)*a[k]+t*b[k] if torch.is_floating_point(a[k]) else a[k]) for k in a}
@torch.no_grad()
def curve(a,b,name):
 m=M().to(D);r=[]
 for t in np.linspace(0,1,21):m.load_state_dict(mix(a,b,float(t)));u,v=eva(m,vl);r.append([name,t,u,v])
 return r
mid=DE[-2];sa=torch.load(K/f'sgd_e{mid}.pt',map_location='cpu');sb=torch.load(K/f'sgd_e{E}.pt',map_location='cpu');ab=torch.load(K/f'adamw_e{E}.pt',map_location='cpu');it=pd.DataFrame(curve(sa,sb,'same')+curve(sb,ab,'cross'),columns=['path','alpha','loss','accuracy']);it.to_csv(C/'interpolation.csv',index=False);plt.figure();
for n0,q0 in it.groupby('path'):plt.plot(q0.alpha,q0.loss,label=n0)
plt.legend();plt.savefig(G/'interpolation.png',dpi=160);plt.close()
summary={'epochs':E,'diagnostic_epochs':DE,'effective_rank_final':{l:er(sf[l]) for l in L},'cka_sgd_adamw':{l:ck(sf[l],af[l]) for l in L},'local_dim90_mean':float(np.mean(ld))};(J/'summary.json').write_text(json.dumps(summary,indent=2));assert not any(R.rglob('*.pt'));print('done',R)
